# AGENTES LLM

## Libraries

In [20]:
import os
from dotenv import load_dotenv
import datetime
import time
import requests

import warnings
warnings.filterwarnings('ignore')

# Gemini
import google.generativeai as gemini


# LangChain
# Configuração de debug do LangChain
from langchain_core.globals import set_debug

# # Modelos LLM (Large Language Models)
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.language_models.chat_models import BaseChatModel

from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

from langchain_core.output_parsers import StrOutputParser


# Criação e execução de agentes
from langchain_classic.agents import( 
Tool, 
AgentExecutor,
create_tool_calling_agent,
create_react_agent)


# # Ferramentas customizadas para agentes
from langchain.tools import tool
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_experimental.tools.python.tool import PythonAstREPLTool

from langchain_classic.memory import ConversationBufferMemory



#ferramentas wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

import wikipedia
from requests.exceptions import JSONDecodeError as ReqJSONDecodeError

wikipedia.set_lang("pt")  # ou "en", conforme sua necessidade


# # Informações e manipulação da execução do interpretador Python
# import sys

# # Expressões regulares
# import re

# # Manipulação de datas e horários
# import datetime

# # Execução de funções assíncronas
# import asyncio

# # Conexões seguras e cliente HTTP assíncrono
# import ssl
# import httpx

# # Manipulação de arquivos e diretórios de forma independente do sistema operacional
# from pathlib import Path

# # Identificadores únicos universais (UUID)
# from uuid import uuid4

# # Manipulação de CSV
# import pandas as pd

# # Tipagem estática (anotações e tipos auxiliares)
# from typing import Any, List, Union, TypedDict

# # Manipulação de XML
# from xml.etree import ElementTree as ET

# # Envio de e-mails (SMTP e MIME)
# from smtplib import SMTP, SMTP_SSL
# from email.mime.base import MIMEBase
# from email.mime.multipart import MIMEMultipart
# from email.mime.text import MIMEText
# from email.utils import formataddr
# try:
#     from email import encoders
# except ImportError:
#     from email import Encoders as encoders

# # Leitura de variáveis de ambiente a partir de arquivos `.env`
# from dotenv import load_dotenv

# # Requisições HTTP (síncronas)
# import requests

# # Exibição de gráficos e imagens
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg

# # =======================
# # OpenAI
# # =======================





# # Construção de prompts
# from langchain_core.messages import HumanMessage



# # Parsers de saída

# # Runnables
# from langchain_core.runnables import RunnableLambda

# #from langchain_classic.agents import Tool, AgentExecutor, create_tool_calling_agent, create_react_agent


# # Criação e execução de agentes
# from langchain_classic.agents import( 
# Tool, 
# AgentExecutor,
# create_tool_calling_agent,
# create_react_agent)

# # from langchain.agents.agent import AgentExecutor
# # from langchain.agents.tool_calling import create_tool_calling_agent

# # Ferramentas
# from langchain_core.tools import Tool, tool
# from langchain_community.agent_toolkits.load_tools import load_tools
# from langchain_experimental.tools.python.tool import PythonAstREPLTool




# # # Tipos de memória utilizados em agentes
# from langchain_classic.memory import ConversationBufferMemory

# # # Componentes de RAG (Retrieval-Augmented Generation)
# from langchain_chroma import Chroma  # Armazenamento vetorial
# from langchain_openai.embeddings import OpenAIEmbeddings  # Embeddings

# from langchain_text_splitters import RecursiveCharacterTextSplitter,MarkdownHeaderTextSplitter
# from langchain_community.document_loaders import PyPDFDirectoryLoader  # Leitura de documentos PDF

# # # Acesso ao hub LangChain de prompts prontos (https://smith.langchain.com/hub)
# from langchain_classic import hub

# # # =======================
# # # MCP (Model Context Protocol)
# # # =======================

# # # Servidor MCP
# from mcp.server.fastmcp import FastMCP

# # # Cliente MCP multi-servidor
# from langchain_mcp_adapters.client import MultiServerMCPClient

# # # =======================
# # # LangGraph
# # # =======================

# # # Criação de agentes com LangGraph
# from langgraph.prebuilt import create_react_agent as create_react_agent_graph

# # # Sistema de checkpoint em memória
# from langgraph.checkpoint.memory import InMemorySaver

# # # Definição e execução de grafos
# from langgraph.graph import StateGraph, END

# # # =======================
# # # Outros
# # # =======================

# # # Ignora avisos durante a execução
# from IPython import get_ipython




In [2]:
# 1. Configuração do diretório de saída
OUTPUT_DOCUMENTS_DIR: str = './documentos/'
os.makedirs(OUTPUT_DOCUMENTS_DIR, exist_ok=True)

# 2. Carregamento das variáveis de ambiente (.env)
ENV_PATH: str = '/home/akel/PycharmProjects/InsurMinds2026/.env'

def carrega_variaveis_ambiente() -> None:
    if os.path.exists(ENV_PATH):
        load_dotenv(ENV_PATH, override=True)
        print("✔ Variáveis de ambiente carregadas do arquivo .env")
    else:
        print(f"⚠ Aviso: Arquivo {ENV_PATH} não foi encontrado no diretório atual.")

carrega_variaveis_ambiente()

✔ Variáveis de ambiente carregadas do arquivo .env


In [3]:
#Carregando chave API
api_key = os.getenv('GOOGLE_API')
gemini.configure(api_key=api_key)

# for model in gemini.list_models():
#     if 'generateContent' in model.supported_generation_methods:
#         print(model.name)

## 1. O que é um Agente?

Um agente é qualquer **entidade** que pode:
* **Perceber** seu ambiente (ex: através de sensores)
* **Processar** essa percepção
* **Agir** sobre o ambiente (através de atuadores).

#### Exemplo 1

In [4]:
# Criando o agente ( recebe pergunta e responde pergunta)

def agente_manual_gemini(pergunta: str) -> str:
    
    model_name: str = "gemini-3.1-flash-lite-preview"
    prompt: str = f"""
    Você é um assistente inteligente com acesso a duas ferramentas:
    1. Calculadora
    2. Wikipedia
    
    Dado a pergunta abaixo, diga o que pretende fazer.
    
    Pergunta: {pergunta}
    
    Responda no formato:
     
    Ação      : [Calculadora|Wikipedia|Responder diretamente]
    Motivo    : ...
    Resultado : ...
    """

    # Inicializa o modelo
    model = gemini.GenerativeModel(model_name)
    response = model.generate_content(
        contents=[
            {
                "role": "user",
                "parts": [prompt]
            }
        ]
    )

    return response.text

In [5]:
pergunta_1="Qual a raiz quadrada de 256?"
pergunta_2="Qual a população de Uruaçu?"
pergunta_3="traduza:\
From their historical beginnings as places to keep the business, legal, historical, and religious records of a civilization,\
libraries have emerged since the middle of the 20th century as a far-reaching body of information resources and services that\
do not even require a building. Rapid developments in computers, telecommunications, and other technologies have made it possible\
to store and retrieve information in many different forms and from any place with a computer and a telephone connection.\
The terms digital library and virtual library have begun to be used to refer to the vast collections of information to which\
people gain access over the Internet, cable television, or some other type of remote electronic connection."


resposta_1 = agente_manual_gemini(pergunta_1)
resposta_2 = agente_manual_gemini(pergunta_2)
resposta_3 = agente_manual_gemini(pergunta_3)

print("Resposta 1 Gemini")
print(resposta_1)
print('\n=================')

print("Resposta 2 Gemini")
print(resposta_2)
print('\n=================')

print("Resposta 3 Gemini")
print(resposta_3)


Resposta 1 Gemini
Ação      : Calculadora
Motivo    : A pergunta solicita uma operação matemática (raiz quadrada) que requer precisão numérica.
Resultado : 16

Resposta 2 Gemini
Ação      : Wikipedia
Motivo    : A população de um município é um dado geográfico e demográfico que consta em bases de dados de conhecimento geral, como a enciclopédia Wikipedia, a partir de censos do IBGE.
Resultado : [Aguardando execução da busca]

Resposta 3 Gemini
Ação      : Responder diretamente
Motivo    : A solicitação consiste em uma tradução de texto, tarefa que posso realizar diretamente utilizando meu conhecimento linguístico, sem a necessidade de cálculos ou consultas a fontes externas.
Resultado : 

"Desde seus primórdios históricos como locais para guardar registros comerciais, legais, históricos e religiosos de uma civilização, as bibliotecas emergiram, desde meados do século XX, como um conjunto abrangente de recursos e serviços de informação que nem sequer exigem um prédio. O rápido desenvolv

---
## 2. LangChain

O **LangChain** é um framework em **Python (e JS)** criado para construção de aplicações que usam LLMs (Large Language Models) como ChatGPT, Claude, Mistral, Llama, etc.


**Porque utilizar:**

* Ele facilita a criação de pipelines, chatbots, assistentes, agentes, RAGs (Retrieval-Augmented Generation), entre outros.

* Facilita a portabilidade entre as LLMs. (Cada LLM tem a sua API com as sua particularidade).

* Ele tem componentes prontos e bem separados (LLMs, Memory, Tools, Chains, Agents).

* Facilita a construção de pipelines complexas.

* Já vem com recursos de tracking, observability, serialization, etc.

* Você pode usar só as partes que quiser (não é obrigatório usar tudo).

Para mais detalhes acesse: https://www.langchain.com/


### 2.1 LCEL- LangChain Expression Language - 

 **LCEL (LangChain Expression Language)** é uma ferramenta poderosa do LangChain projetada para facilitar a construção de cadeias de chamadas (chains) de forma fluida e eficiente. Pense nele como a "cola" que une diferentes componentes de um aplicativo de IA, como modelos de linguagem (LLMs), prompts e ferramentas, em um fluxo de trabalho coerente.

A grande é sua capacidade de permitir que os desenvolvedores criem pipelines complexos mais simples usando o operador | (pipe), semelhante ao que se usa em shells como Bash. Isso torna a leitura e a escrita das cadeias muito mais intuitiva.

O LCEL traz consigo uma série de benefícios importantes:

* **Streaming**: Ele suporta o streaming de tokens, ou seja, as respostas são geradas em tempo real, em vez de esperar a conclusão total da cadeia. Isso melhora a experiência do usuário, pois a resposta começa a aparecer imediatamente.

* **Paralelismo**: O LCEL executa operações que não dependem umas das outras em paralelo automaticamente, o que melhora o desempenho da sua aplicação.

* **Fallback**: Ele permite a definição de mecanismos de "fallback", onde você pode configurar um plano B caso um modelo ou ferramenta falhe, aumentando a robustez da sua aplicação.

* **Composição**: A facilidade de combinar e reutilizar diferentes partes da sua cadeia, tornando o código mais modular e fácil de manter.

* **Acessibilidade**: Suporte para chamadas síncronas e assíncronas, permitindo que você adapte o código ao seu ambiente.

#### Como o LCEL funciona?
O LCEL é baseado no encadeamento de objetos que implementam a interface **Runnable**. Componentes de uma cadeia no LangChain (PromptTemplate, ChatModel, OutputParser) são Runnables.

> A sintaxe do LangChain utiliza o operador **|** para encadear componentes. No exemplo `prompt | model`, o processamento ocorre sequencialmente:
>
> 
> 1. **Entrada:** A cadeia recebe os dados iniciais.
> 2. **`prompt`:** Formata a entrada em um *PromptValue*.
> 3. **`model`:** O resultado do prompt é passado como entrada para o model (o LLM). O modelo, por sua vez, gera uma ChatMessage.
> 4. **Saída:** O resultado do model é a saída da cadeia. necessario converter em string (strOutputParser)
>    
>  Exemplo:
>  ### **`prompt` | `modelo`  | `StrOutputParser()`**
>  *StrOutputParser()* Converte a saída do modelo de linguagem (que originalmente é um objeto AIMessage) em uma string de texto puro.

Vamos a um exemplo prático:

In [6]:
llm_gemini = ChatGoogleGenerativeAI(temperature=0, model="gemini-3.1-flash-lite-preview",google_api_key=os.getenv("GOOGLE_API"))
set_debug(False)

#### Exemplo 2

In [7]:
# exemplo_02.py

# Criando o agente
def agente_lcel(pergunta:str) -> str:
    modelo:str = llm_gemini # Gemini
    prompt:str = ChatPromptTemplate.from_messages(
        [
            ("system", "Você é um assistente inteligente com acesso a 2 ferramentas:"),
            ("system", "1. Calculadora"),
            ("system", "2. Wikipedia"),
            ("system", "Dado a pergunta abaixo, diga o que pretende fazer."),
            ("human", "{pergunta}"),                                          # pergunta aqui
            ("system", "Responda no formato:"),
            ("system", "Ação      : [Calculadora|Wikipedia|Responder diretamente]"),
            ("system", "Motivo    : ..."),
            ("system", "Resultado : ..."),
        ])

    # LCEL-LangChain Expression Language
    cadeia = prompt | modelo  | StrOutputParser() 
    return cadeia.invoke({"pergunta": pergunta})

In [8]:
respostal_cel1 = agente_lcel(pergunta_1)
respostal_cel2 = agente_lcel(pergunta_2)
respostal_cel3 = agente_lcel(pergunta_3)

print("Resposta 1 LCEL")
print(respostal_cel1 )
print('\n=================')

print("Resposta 2 LCEL")
print(respostal_cel2 )
print('\n=================')

print("Resposta 3 LCEL")
print(respostal_cel3 )
print('\n=================')


Resposta 1 LCEL
Ação      : Calculadora
Motivo    : Preciso realizar a operação matemática de extração da raiz quadrada do número 256.
Resultado : 16

Resposta 2 LCEL
Ação      : Wikipedia
Motivo    : Preciso consultar a base de dados da Wikipedia para obter o dado populacional mais recente e confiável sobre o município de Uruaçu.
Resultado : [Aguardando consulta]

Resposta 3 LCEL
Ação      : Responder diretamente
Motivo    : O texto fornecido é um trecho informativo sobre a evolução das bibliotecas e não requer consultas externas (cálculos ou busca de fatos) para ser traduzido.
Resultado : 

"Desde seus primórdios históricos como locais para guardar registros comerciais, legais, históricos e religiosos de uma civilização, as bibliotecas emergiram, desde meados do século XX, como um corpo abrangente de recursos e serviços de informação que nem sequer exigem um prédio. O rápido desenvolvimento dos computadores, das telecomunicações e de outras tecnologias tornou possível armazenar e rec

### 2.2 AgentExecutor


O **AgentExecutor** é o motor de execução de um agente. Ele é a lógica de alto nível que orquestra o processo de tomada de decisão. As principais responsabilidades do AgentExecutor são:

* **Observar o Histórico de Conversas**: Ele recebe o prompt do usuário e o histórico da conversa.

* **Chamar o Agent**: Ele envia essa informação para o Agent (que é um Runnable, ou seja, pode ser construído com LCEL). O Agent é a "**mente**" que decide a próxima ação.

* **Processar a Resposta do Agente**: A resposta do Agent pode ser uma de duas coisas:
> * **Uma AgentAction**: O agente decidiu usar uma ferramenta. O AgentExecutor então chama a ferramenta especificada com a entrada correta.
> * **Uma AgentFinish**: O agente decidiu que a tarefa está completa e tem a resposta final para o usuário.

* **Loop**: Se for uma **AgentAction**, o `AgentExecutor` executa a ferramenta, obtém o resultado e repete o processo (volta para o **passo 1**), enviando o resultado da ferramenta de volta para o agente. Ele faz isso em um loop até que o agente decida que a tarefa está finalizada (**AgentFinish**).

Resumindo o `AgentExecutor` é a camada que gerencia o ciclo de vida do agente, o "ciclo de raciocínio".


#### **Qual é melhor utilizar o `LCEL` ou o `AgentExecutor`?**

Você não precisa escolher entre LCEL e AgentExecutor. O AgentExecutor é, na verdade, uma implementação de uma cadeia construída com LCEL, porém com uma lógica de alto nível para gerenciar o ciclo de vida do agente.


#### Exemplo 2.1: LLM + Prompt

In [9]:
def string_gemini(out_agent_exe):
    if isinstance(out_agent_exe, list) and len(out_agent_exe) > 0:
        if isinstance(out_agent_exe[0], dict) and 'text' in out_agent_exe[0]:
            return out_agent_exe[0]['text']
    
    return  out_agent_exe[0]['text']
        


def agente_langchain(pergunta:str) -> dict:
    modelo = llm_gemini
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "Você é um assistente inteligente com acesso a duas ferramentas:"),
            ("system", "1. Calculadora"),
            ("system", "2. Wikipedia"),
            ("system", "Dado a pergunta abaixo, diga o que pretende fazer."),
            ("human", "{pergunta}"),
            ("system", "Responda no formato:"),
            ("system", "Ação      : [Calculadora|Wikipedia|Responder diretamente]"),
            ("system", "Motivo    : ... "),
            ("system", "Resultado : ... "),
            MessagesPlaceholder(variable_name="agent_scratchpad"),                        #fluxo de raciocínio da LLM
        ])

    minhas_ferramentas = []
    agente = create_tool_calling_agent(modelo, tools=minhas_ferramentas , prompt=prompt)        # Tomar decisão
    executor_do_agente = AgentExecutor(agent=agente, tools=minhas_ferramentas ) 
    # Executar a decisão
    resposta = executor_do_agente.invoke({"pergunta": pergunta})

    resposta=string_gemini(resposta['output'])   # aqui não tem o StrigOutParser. portanto, para o gemini é necessario uma função para pega o dic
    return resposta
    

In [10]:
respostal_lc1 = agente_langchain(pergunta_1)
respostal_lc2 = agente_langchain(pergunta_2)
respostal_lc3 = agente_langchain(pergunta_3)

print("Resposta 1 LC")
print(respostal_lc1 )
print('\n=================')

print("Resposta 2 LC")
print(respostal_lc2 )
print('\n=================')

print("Resposta 3 LC")
print(respostal_lc3 )
print('\n=================')


Resposta 1 LC
Ação      : Calculadora
Motivo    : Preciso realizar a operação matemática de extração da raiz quadrada do número 256.
Resultado : 16

Resposta 2 LC
Ação      : Wikipedia
Motivo    : Preciso consultar a base de dados da Wikipedia para obter o dado populacional mais recente e confiável sobre o município de Uruaçu.
Resultado : ...

Resposta 3 LC
Ação      : Responder diretamente
Motivo    : O texto fornecido é um trecho informativo sobre a evolução das bibliotecas e não requer consultas externas para ser traduzido.
Resultado : 

"Desde seus primórdios históricos como locais para guardar registros comerciais, legais, históricos e religiosos de uma civilização, as bibliotecas emergiram, desde meados do século XX, como um corpo abrangente de recursos e serviços de informação que nem sequer exigem um prédio. O rápido desenvolvimento dos computadores, das telecomunicações e de outras tecnologias tornou possível armazenar e recuperar informações em muitas formas diferentes e a pa

#### Exemplo 2.2: LLM + Tool + Prompt
> ver detalhes sobre ferramentas\
 https://python.langchain.com/docs/integrations/tools/ \
 https://python.langchain.com/docs/versions/migrating_chains/llm_math_chain/


In [11]:
def agente_langchain2(pergunta:str) -> dict:

    modelo = llm_gemini
    
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", "Você é um agente responsável por explicar passo a passo solução de problemas matemáticos."),
            ("system", "REGRAS OBRIGATÓRIAS DE EXECUÇÃO:"),
            ("system", "1. Para qualquer cálculo, você DEVE acionar uma das ferramentas (Tools) registradas."),
            ("system", "2. NÃO resolva, não deduza e não faça cálculos por conta própria sem invocar uma ferramenta."),
            ("system", "3. Se a pergunta exigir uma operação que NENHUMA das ferramentas disponíveis consiga realizar "),
            ("system","responda EXATAMENTE:Não é possível realizar esta operação com as ferramentas disponíveis."),
            ("system", "4. Apenas responda com a solução se ela for obtida diretamente através do retorno de uma ferramenta."),
            ("system", "A Explicação deve seguir o formato abaixo:"),
            ("system", "Explicar sobre o que se trata o problema."),
            ("system", "Ferramenta utilizadas: . "),
            ("system", "Caso exista solução: seguir passoas abaixo, do contrário explicar o motivo de não exibir solução"),
            ("system", "A solução terá n passos."),
            ("system", "Passo 1: nome do passo"),
            ("system", "detalhe do  passo 1 ..."),
            ("system", "Passo 2: nome do passo"),
            ("system", "detalhe do  passo 2..."),
            ("system", "..."),
            ("system", "Resultado Final :"),
            ("system", "..."),
            ("human", "{input}"),

            MessagesPlaceholder(variable_name="agent_scratchpad"), # Onde o agente irá escrever suas anotações (Pensamento)
        ])
    
    #ferramentas =load_tools(["llm-math"], llm=modelo)
    ferramentas = [PythonAstREPLTool()]           #mais robusto para calculo simbolico que llm-math
    
    agente = create_tool_calling_agent(modelo, ferramentas, prompt)
    
    executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas,return_intermediate_steps=True)  

    resposta = executor_do_agente.invoke({"input": pergunta})
    
    # 2. Imprima os passos intermediários para verificar
    passos = resposta.get("intermediate_steps", [])
    print(f"\n--- Ferramentas chamadas: {len(passos)} ---")
    for acao, resultado in passos:
        print(f" Ferramenta usada: {acao.tool}")
        print(f" Entrada enviada: {acao.tool_input}")
        print(f" Resposta da ferramenta: {resultado}")
    print("-----------------------------------\n")
    

    resposta=string_gemini(resposta['output'])


    return resposta

In [12]:
agente_langchain2 

pergunta0="Qual é a raiz quadrada de 169 vezes 2?"
pergunta1="Mostre a solução da equação x^2-2x+6= 0"
pergunta2="Mostre a solução da equação x^3-4x^2 +3x -4 = 0"
pergunta3="Mostre a solução da equação x^3-5x^2 +2x +8= 0"
pergunta4="Mostre a solução da integral 2x/(1+x^2)"
pergunta5="Mostre a solução da integral 1/(9+x^2)^0.5"

resposta_LC2 = agente_langchain2(pergunta0)
print("Resposta  LC2")
print(resposta_LC2 )
print('\n=================')



--- Ferramentas chamadas: 1 ---
 Ferramenta usada: python_repl_ast
 Entrada enviada: {'query': 'import math\nresultado = math.sqrt(169) * 2\nprint(resultado)\n'}
 Resposta da ferramenta: 26.0

-----------------------------------

Resposta  LC2
Este problema trata do cálculo da raiz quadrada de um número (169) seguido pela multiplicação do resultado por 2.

Ferramenta utilizada: `python_repl_ast`

A solução terá 2 passos.

Passo 1: Calcular a raiz quadrada de 169.
A raiz quadrada de 169 é 13, pois 13 * 13 = 169.

Passo 2: Multiplicar o resultado obtido por 2.
Multiplicando 13 por 2, obtemos 26.

Resultado Final: 26



### 2.3 Ferramentas (Tools)

No contexto do LangChain, as **ferramentas** (ou tools) são funções que um modelo de linguagem (LLM) pode chamar para interagir com o mundo exterior. Pense nelas como os "**sentidos**" e "**mãos**" do seu agente de IA.

Alguns exemplos de ferramentas comuns incluem:

* **Busca na internet**: Uma ferramenta que usa um buscador como o Google ou o DuckDuckGo para encontrar informações atualizadas.
* **Calculadora**: Uma ferramenta que executa operações matemáticas precisas.
* **API de clima**: Uma ferramenta que faz uma chamada a uma API para obter a previsão do tempo para uma cidade.
* **Leitor de arquivos**: Uma ferramenta que permite ao agente ler o conteúdo de um documento.
* **Ferramenta de SQL**: Uma ferramenta que executa consultas em um banco de dados.

#### Exemplo 2.3: LLM+ tools( by def function)

In [13]:

@tool
def get_current_time(*args, **kwargs) -> str:
    """O objetivo dessa ferramenta é retornar a data e hora atual."""
    now = datetime.datetime.now()
    return f"A data e hora atual é {now.strftime('%Y-%m-%d %H:%M:%S')}"

def agente_langchain3(pergunta:str,usar_ferramentas:bool=True) -> dict:
    
    modelo=llm_gemini
    
    prompt = ChatPromptTemplate.from_messages([
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad"), # Onde o agente irá escrever suas anotações (Pensamento)
        ])

    ferramentas = [get_current_time] if usar_ferramentas else []
   #ferramentas terá get_current_time se True. 
   #ou
   #se usar_ferramentas for verdadeiro -add get_current em current_time

    agente = create_tool_calling_agent(modelo, ferramentas, prompt)
    executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas,return_intermediate_steps=True)
    resposta = executor_do_agente.invoke({"input": pergunta})
    
    resposta=string_gemini(resposta['output'])
    return resposta


In [14]:
perguntaA="Qual é a data inicial e final dessa semana?"
perguntaB="Em que dia da semana cai daqui a 38 dias?"
perguntaC="Em que dia da semana cai daqui a 38 dias?"

resposta_LC3a = agente_langchain3(perguntaB,False)
resposta_LC3b = agente_langchain3(perguntaB,True)
print("Resposta  LC3")
print(resposta_LC3a )
print('\n=================')
print(resposta_LC3b )
resposta_LC3b = agente_langchain3(perguntaC,True)


Resposta  LC3
Para saber que dia da semana será daqui a 38 dias, basta dividir 38 por 7 (o número de dias da semana) e ver o resto:

1.  38 ÷ 7 = 5 semanas completas e **sobra 3 dias** (pois 7 × 5 = 35, e 38 - 35 = 3).
2.  Isso significa que daqui a 38 dias será o mesmo dia da semana de hoje, mais 3 dias.

**Como hoje é terça-feira, dia 22 de outubro de 2024:**
*   +1 dia: quarta-feira
*   +2 dias: quinta-feira
*   **+3 dias: sexta-feira**

Portanto, daqui a 38 dias será uma **sexta-feira** (dia 29 de novembro de 2024).

Hoje é dia 10 de agosto de 2026, uma segunda-feira.

Para saber que dia da semana será daqui a 38 dias, podemos dividir 38 por 7 (o número de dias em uma semana):

*   38 ÷ 7 = 5 semanas completas e **3 dias de resto**.

Contando 3 dias a partir de segunda-feira:
1. Terça-feira
2. Quarta-feira
3. **Quinta-feira**

Portanto, daqui a 38 dias será uma **quinta-feira** (dia 17 de setembro de 2026).


### 2.4 Memória

A **memória** (ou memory) no LangChain é o componente que permite que os agentes e as cadeias de conversa retenham informações de interações anteriores. Sem a memória, cada interação seria tratada como uma nova e isolada, fazendo com que o LLM "**esquecesse**" o que foi dito nos turnos anteriores.

A memória é crucial para construir chatbots e assistentes que podem ter conversas fluidas e contextuais. Ela injeta o histórico da conversa no prompt de cada nova chamada ao LLM, permitindo que o modelo use esse contexto para gerar respostas mais relevantes.

Existem vários tipos de memória no LangChain, cada um com uma estratégia diferente para armazenar e recuperar o histórico:

* **ConversationBufferMemory**: A forma mais simples de memória. Ela armazena todas as mensagens da conversa em uma variável e as injeta no prompt. É fácil de usar, mas pode se tornar ineficiente para conversas muito longas, pois o tamanho do prompt cresce.

* **ConversationBufferWindowMemory**: Similar à anterior, mas armazena apenas as últimas N interações (uma "janela" de conversa). Isso evita que o prompt fique grande demais, mantendo apenas o contexto mais recente.

* **ConversationSummaryMemory**: Em vez de armazenar a conversa inteira, ela cria um resumo contínuo das interações anteriores. Isso é ótimo para conversas longas, pois mantém o contexto sem sobrecarregar o prompt.

* **ConversationSummaryBufferMemory**: Uma combinação das duas últimas, que armazena as interações recentes na íntegra e resume as interações mais antigas.


#### Exemplo 2.4: LLM+ tools+ memory

In [15]:
# aqui é um pouco diferente dos codigos acima.

@tool
def get_current_time(*args, **kwargs) -> str:
    """O objetivo dessa ferramenta é retornar a data e hora atual."""
    now = datetime.datetime.now()
    return f"A data e hora atual é: {now.strftime('%Y-%m-%d %H:%M:%S')}"


def agente_langchain4(usar_ferramentas:bool=True) -> dict:
    
    modelo=llm_gemini
    
    memoria = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
    
    prompt = ChatPromptTemplate.from_messages(
        [
            MessagesPlaceholder(variable_name="chat_history"), # O placeholder para o histórico
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad"), # Onde o agente irá escrever suas anotações (Pensamento)
        ])

    ferramentas = [get_current_time] if usar_ferramentas else []

    agente = create_tool_calling_agent(modelo, ferramentas, prompt)
    executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas,memory=memoria,return_intermediate_steps=True)
    
    #resposta = executor_do_agente.invoke({"input": pergunta})
    #resposta=string_gemini(resposta['output'])
    return executor_do_agente


In [16]:
executor_do_agente = agente_langchain4(True)
apresentacao="Meu nome é alberto akel"
perguntaA="Qual é a data inicial e final dessa semana?"
perguntaB="Qual meu sobrenome"
perguntaC="Qual foi minha ultima pergunta?"

executor_do_agente.invoke({"input": apresentacao})
resposta_a = executor_do_agente.invoke({"input": perguntaA})
resposta_a=string_gemini(resposta_a['output'])
print(resposta_a)
print('\n' + '=' * 10)

resposta_b = executor_do_agente.invoke({"input": perguntaB})
resposta_b=string_gemini(resposta_b['output'])
print(resposta_b)

print('\n' + '=' * 10)

resposta_c = executor_do_agente.invoke({"input": perguntaC})
resposta_c=string_gemini(resposta_c['output'])
print(resposta_c)


/tmp/ipykernel_17909/1495514091.py:14: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memoria = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


Considerando que hoje é segunda-feira, 10 de agosto de 2026, esta semana compreende o seguinte período:

*   **Data inicial:** 10 de agosto de 2026 (segunda-feira)
*   **Data final:** 16 de agosto de 2026 (domingo)

O seu sobrenome é **Akel**.

Sua última pergunta foi: "Qual meu sobrenome".


### 2.5 Prompt

O "prompt" é a instrução, pergunta ou texto inicial que você fornece a um modelo de linguagem (LLM) para que ele gere uma resposta. 

**Tipos de Prompts** \
No contexto do LangChain e do desenvolvimento com LLMs, as duas categorias mais importantes são:

* **Prompts Simples (Strings)**: É uma string de texto simples que você envia diretamente para o modelo. Não há formatação complexa ou variáveis.
  
* **Prompts Estruturados (Templates)**: Este tipo de prompt é uma estrutura reutilizável, ou um template, que contém espaços reservados para variáveis. Em vez de escrever o prompt completo a cada vez, você preenche essas variáveis com dados dinâmicos. Essa abordagem é a mais utilizada em aplicações reais, pois permite criar prompts robustos e flexíveis.

 #### Exemplo 2.5: PromptTemplate

In [17]:
# Criando o agente


def agente_langchain5(pergunta:str) -> str:
    
    modelo=llm_gemini
    ferramentas=[]
    
    prompt = PromptTemplate(
        input_variables=["pergunta", "agent_scratchpad"],
        template= """ Você é um assistente inteligente com acesso a duas ferramentas:
                        1. Calculadora
                        2. Wikipedia
                    Dado a pergunta abaixo, diga o que pretende fazer.
                    Pergunta: {pergunta}
                    {agent_scratchpad}
                    Responda no formato:
                    Ação      : [Calculadora|Wikipedia|Responder diretamente]
                    Motivo    : ...
                    Resultado : ...
        """)

    
    agente = create_tool_calling_agent(modelo, tools=ferramentas, prompt=prompt)
    executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas)
    resposta = executor_do_agente.invoke({"pergunta": pergunta})
    resposta=string_gemini(resposta['output'])

    return resposta

In [18]:
respostal_lc1 = agente_langchain5(pergunta_1)
respostal_lc2 = agente_langchain5(pergunta_2)
respostal_lc3 = agente_langchain5(pergunta_3)

print("Resposta 1 LC5")
print(respostal_lc1 )
print('\n=================')

print("Resposta 2 LC5")
print(respostal_lc2 )
print('\n=================')

print("Resposta 3 LC5")
print(respostal_lc3 )
print('\n=================')


Resposta 1 LC5
Ação      : Calculadora
Motivo    : A pergunta requer uma operação matemática (cálculo de raiz quadrada) para ser respondida com precisão.
Resultado : 16

Resposta 2 LC5
Ação      : Wikipedia
Motivo    : Preciso consultar uma fonte de dados confiável e atualizada sobre a demografia do município de Uruaçu para responder à pergunta.
Resultado : [Aguardando consulta à Wikipedia]

Resposta 3 LC5
Ação      : Responder diretamente
Motivo    : A tarefa consiste em uma tradução de texto, que não requer cálculos matemáticos nem consultas a enciclopédias, sendo uma habilidade inerente ao modelo de linguagem.
Resultado : Desde os seus primórdios históricos como locais para guardar registros comerciais, legais, históricos e religiosos de uma civilização, as bibliotecas emergiram, desde meados do século XX, como um corpo abrangente de recursos e serviços de informação que nem sequer exigem um edifício. O rápido desenvolvimento dos computadores, das telecomunicações e de outras tecnol

 #### Exemplo 2.6: ChatPromptTemplate

In [21]:
#configuração wikipedia

_session = requests.Session()
_session.headers.update({"User-Agent": "agente_langchain7/1.0 (contato: albertoakell@gmail.com)"})

def _buscar_wikipedia_raw(query: str, lang: str = "pt", timeout: int = 10) -> dict | None:
    base_url = f"https://{lang}.wikipedia.org/w/api.php"


    #1-busca o título
    search_params = {
        "action": "query", "list": "search",
        "srsearch": query, "format": "json", "srlimit": 1     #termo de busca ("srlimit-> limita apenas 1 resultado na busca)
    }
    try:
        r = _session.get(base_url, params=search_params, timeout=timeout)
        r.raise_for_status()
        data = r.json()
    except (requests.exceptions.RequestException, requests.exceptions.JSONDecodeError) as e:
        print(f"[wikipedia] Erro na busca: {type(e).__name__}: {e}")
        return None

    resultados = data.get("query", {}).get("search", [])
    if not resultados:
        return None
    titulo = resultados[0]["title"]
    
    #2-busca o resumo daquele título(ou query)
    extract_params = {
        "action": "query", "prop": "extracts",
        "exintro": True, "explaintext": True,
        "titles": titulo, "format": "json"
    }
    try:
        r = _session.get(base_url, params=extract_params, timeout=timeout)
        r.raise_for_status()
        data = r.json()
    except (requests.exceptions.RequestException, requests.exceptions.JSONDecodeError) as e:
        print(f"[wikipedia] Erro ao buscar extrato: {type(e).__name__}: {e}")
        return None

    paginas = data.get("query", {}).get("pages", {})
    if not paginas:
        return None
    pagina = next(iter(paginas.values()))
    extrato = pagina.get("extract", "")
    return {"titulo": titulo, "resumo": extrato[:1000]}


@tool
def buscar_wikipedia(query: str) -> str:
    """Busca um resumo de um tópico na Wikipedia em português. Retorna o título da página e um resumo."""
    resultado = _buscar_wikipedia_raw(query)
    if resultado is None:
        return f"Nenhum resultado encontrado na Wikipedia para: '{query}'"
    return f"Page: {resultado['titulo']}\nSummary: {resultado['resumo']}"


In [22]:
def agente_langchain6(pergunta:str, tentativas:int=5) -> str:
    
    for tentativa in range(tentativas):
        try:
            inicio = time.time()  # Marca o tempo inicial
            modelo=llm_gemini
    
            ferramentas=[buscar_wikipedia]

            prompt=ChatPromptTemplate.from_messages([
                ('system', """Você é um assistente inteligentes com acesso a duas ferramentas:
                        1. Calculadora
                        2. wikipedia

                    REGRAS OBRIGATÓRIAS:
                    - Toda informação factual na sua resposta final DEVE vir literalmente do conteúdo 
                    retornado pelas ferramentas (Wikipedia ou Calculadora). Não é permitido completar, 
                    estimar ou inferir dados (datas, números, estatísticas, nomes) que não estejam 
                    explicitamente no resultado da ferramenta, mesmo que você "saiba" a resposta.
                    - Se a ferramenta não retornar a informação pedida, tente reformular a busca no 
                    máximo mais 1 vez. Se ainda assim não encontrar, responda explicitamente:
                    "Não encontrei essa informação nas fontes consultadas."
                    - Nunca cite números, datas ou fatos específicos sem que eles apareçam no texto 
                    retornado pela ferramenta na etapa anterior.
                    
                    Dado a pergunta diga o que pretende fazer no seguinte formato:
                    Ação      : [Calculadora|Wikipedia|Responder diretamente]
                    Motivo    : ...
                    Resultado : ... 

                    No campo Resultado, se a informação não estiver disponível nas ferramentas, 
                    diga isso claramente em vez de inventar um valor.
                    """),
                ("human", "{input}"),
                MessagesPlaceholder(variable_name="agent_scratchpad"),
            ])

            agente = create_tool_calling_agent(modelo, ferramentas, prompt)
            executor_do_agente = AgentExecutor(agent=agente, tools=ferramentas,return_intermediate_steps=True)  
            resposta = executor_do_agente.invoke({"input": pergunta})
            break
        except ReqJSONDecodeError:
            print(f"⚠️ Falha na API da Wikipedia (tentativa {tentativa+1}/{tentativas}), tentando novamente...")
            time.sleep(2 * (tentativa + 1))
    else:
        raise RuntimeError("Wikipedia API falhou após múltiplas tentativas.")

            
    fim = time.time()      # Marca o tempo final
    tempo_total = fim - inicio


     # 2. Imprima os passos intermediários para verificar
    passos = resposta.get("intermediate_steps", [])

    print(f"⏱️ Tempo de resposta: {tempo_total:.2f} segundos")
    print(f"🛠️ Ferramentas chamadas: {len(passos)}")
    
    print(f"\n--- Ferramentas chamadas: {len(passos)} ---")
    for acao, resultado in passos:
        print(f" Ferramenta usada: {acao.tool}")
        print(f" Entrada enviada: {acao.tool_input}")
        print(f" Resposta da ferramenta: {resultado}")
    print("-----------------------------------\n")
    
    resposta=string_gemini(resposta['output'])
    
    return resposta

In [23]:
respostal_lc1 = agente_langchain6(pergunta_1)

print("Resposta 1 LC6")
print(respostal_lc1 )
print('\n=================')

respostal_lc2 = agente_langchain6(pergunta_2)
print("Resposta 2 LC6")
print(respostal_lc2 )
print('\n=================')

respostal_lc3 = agente_langchain6(pergunta_3)
print("Resposta 3 LC6")
print(respostal_lc3 )
print('\n=================')

⏱️ Tempo de resposta: 1.41 segundos
🛠️ Ferramentas chamadas: 0

--- Ferramentas chamadas: 0 ---
-----------------------------------

Resposta 1 LC6
Ação      : Calculadora
Motivo    : Preciso calcular a raiz quadrada de 256.
Resultado : 16

⏱️ Tempo de resposta: 3.00 segundos
🛠️ Ferramentas chamadas: 1

--- Ferramentas chamadas: 1 ---
 Ferramenta usada: buscar_wikipedia
 Entrada enviada: {'query': 'Uruaçu'}
 Resposta da ferramenta: Page: Uruaçu
Summary: Uruaçu é um município brasileiro localizado no estado de Goiás. Sua população, conforme estimativas do censo 2025 do Instituto Brasileiro de Geografia e Estatística (IBGE), é de 44.533 habitantes. O município abriga o Lago de Serra da Mesa, um dos maiores lagos artificiais do Brasil, formado para a geração de energia hidrelétrica.
-----------------------------------

Resposta 2 LC6
Ação      : Wikipedia
Motivo    : Buscar a população de Uruaçu.
Resultado : A população de Uruaçu, conforme estimativas do censo 2025 do Instituto Brasileiro

In [24]:

pergunta_4="Qual a temperatura da coroa Solar"
respostal_lc4= agente_langchain6(pergunta_4)
print("Resposta 3 LC6")
print(respostal_lc4 )
print('\n=================')

⏱️ Tempo de resposta: 3.55 segundos
🛠️ Ferramentas chamadas: 1

--- Ferramentas chamadas: 1 ---
 Ferramenta usada: buscar_wikipedia
 Entrada enviada: {'query': 'coroa solar'}
 Resposta da ferramenta: Page: Coroa estelar
Summary: A coroa estelar é a camada mais externa da atmosfera de uma estrela. Ela é composta de plasma.
A coroa do Sol, também chamada de coroa branca, coroa de Fraunhoffer ou corona, fica acima da cromosfera e se estende por milhões de quilômetros no espaço sideral. Ela é mais facilmente vista durante um eclipse solar total, mas também pode ser observada com um coronógrafo. As medições espectroscópicas indicam forte ionização na coroa e uma temperatura de plasma superior a 1.000.000 Kelvins, muito mais quente do que a superfície do Sol, conhecida como fotosfera.
Corona (latim para "coroa") é, por sua vez, derivado do grego antigo κορώνη (korṓnē) "guirlanda, coroa de flores".
-----------------------------------

Resposta 3 LC6
Ação      : Wikipedia
Motivo    : Buscar in